In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" # Cambia el 1 por el id de la GPU que quieras usar

In [2]:
import torch
torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0)

(True, 1, 'NVIDIA GeForce RTX 3090')

In [3]:
import pandas as pd
from datasets import load_dataset


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
SAMPLE_SIZE = 1000 # Ajustable según capacidad
MAX_LENGTH = 512
SEED = 41
BATCH_SIZE= 4

# Modelos
MODEL_NAMES = {
    'LLaDA_8B': 'GSAI-ML/LLaDA-8B-Base',
    'LLaDA_1_5B': 'GSAI-ML/LLaDA-1.5',
    'Dream_LLaDA_7B': 'Dream-org/Dream-v0-Base-7B'
}


dataset = load_dataset("yaful/MAGE", split="test")
df_full = dataset.to_pandas()
df_sample = df_full.sample(n=SAMPLE_SIZE, random_state=42)


df_sample['Clase_Real'] = df_sample['label'].apply(lambda x: 'Humano' if x == 1 else 'IA')

print("\nPrimeras filas del DataFrame de muestra:")
print(df_sample[['text', 'Clase_Real', 'label']].head())

balance = df_sample['Clase_Real'].value_counts(normalize=True) * 100

print("\n--- Balance de Clases en la Muestra ---")
print(balance.to_string())

df_sample['Clase_Real_Binaria'] = df_sample['label']


df_sample['text_cleaned'] = df_sample['text'].str.replace('\s+', ' ', regex=True).str.strip()

df_sample = df_sample.dropna(subset=['text_cleaned'])

df_sample.reset_index(drop=True, inplace=True)
df_sample['Texto_ID'] = df_sample.index

texts = df_sample['text_cleaned']
labels = df_sample['Clase_Real_Binaria'].values

# Contenedores de resultados
performance_data = []
all_results = []


Primeras filas del DataFrame de muestra:
                                                    text Clase_Real  label
21764  Never again...never again!!' This place is ter...         IA      0
46722  put the carpet on the floor, they measure it, ...     Humano      1
49245  [substeps] You may do this process before you ...     Humano      1
30867  I believe mandatory minimum laws are unjust, c...     Humano      1
10010  Wales coach Warren Gatland has hailed Shane Wi...         IA      0

--- Balance de Clases en la Muestra ---
Clase_Real
IA        51.1
Humano    48.9


In [4]:
# Función de medición de recursos
def resource_wrapper(fn, *args, device='cuda', verbose=True):
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024**3  # en GB

    start_time = time.time()
    try:
        result = fn(*args)
    except Exception as e:
        if verbose:
            print(f"[ERROR] La función {fn.__name__} falló: {e}")
        raise e
    elapsed = time.time() - start_time

    vram_peak = torch.cuda.max_memory_allocated() / 1024**3 if device == 'cuda' else 0.0
    mem_after = process.memory_info().rss / 1024**3
    cpu_mem = mem_after - mem_before

    return result, elapsed, vram_peak, cpu_mem

In [5]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import psutil
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForMaskedLM

# --- CONFIGURACIÓN ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- CARGA DE MODELOS ---
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# LLaDA_8B (Difusión)
tokenizer_LLaDA_8B = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaDA_8B'], trust_remote_code=True)
model_LLaDA_8B = AutoModel.from_pretrained(MODEL_NAMES['LLaDA_8B'], quantization_config=bnb_config, trust_remote_code=True, dtype=DTYPE).eval()
if hasattr(model_LLaDA_8B, "tie_weights"): model_LLaDA_8B.tie_weights()
LLaDA_8B_DEVICE = next(model_LLaDA_8B.parameters()).device

# LLaDA_1_5B (Difusión)
tokenizer_LLaDA_1_5B = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaDA_1_5B'], trust_remote_code=True)
model_LLaDA_1_5B = AutoModel.from_pretrained(MODEL_NAMES['LLaDA_1_5B'], quantization_config=bnb_config, trust_remote_code=True, dtype=DTYPE).eval()
if hasattr(model_LLaDA_1_5B, "tie_weights"): model_LLaDA_1_5B.tie_weights()
LLaDA_1_5B_DEVICE = next(model_LLaDA_1_5B.parameters()).device

# Dream_LLaDA_7B (Difusión)
tokenizer_Dream_LLaDA_7B = AutoTokenizer.from_pretrained(MODEL_NAMES['Dream_LLaDA_7B'], trust_remote_code=True)
model_Dream_LLaDA_7B = AutoModel.from_pretrained(MODEL_NAMES['Dream_LLaDA_7B'], quantization_config=bnb_config, trust_remote_code=True, dtype=DTYPE).eval()
if hasattr(model_Dream_LLaDA_7B, "tie_weights"): model_Dream_LLaDA_7B.tie_weights()
Dream_LLaDA_7B_DEVICE = next(model_Dream_LLaDA_7B.parameters()).device


/opt/conda/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
2026-02-02 17:35:37.030406: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 17:35:37.119448: I tensorflow/core/platform/cpu_feature_guard.cc:210]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### SCORES

In [6]:
def batch_llada_scores(texts, model, tokenizer, batch_size, device):
    scores = []

    is_dream = model.__class__.__name__.lower().startswith("dream")

    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Scores"):
        batch = texts[i:i+batch_size].tolist()

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )

        # 🔥 CLAVE: Dream NO acepta attention_mask 2D
        if not is_dream and "attention_mask" in inputs:
            inputs["attention_mask"] = inputs["attention_mask"].to(torch.bool)
        else:
            inputs.pop("attention_mask", None)

        inputs = inputs.to(device)

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            outputs = model(**inputs)
            logits = outputs.logits[:, :-1, :]
            labels = inputs["input_ids"][:, 1:]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                reduction="none"
            ).view(labels.shape)

            scores.extend((-loss.mean(dim=1)).cpu().tolist())

    return scores


In [7]:
# Inicializamos el DataFrame de métricas con la info básica
df_metrics = df_sample[['text_cleaned', 'label']].copy()
df_metrics['Clase_Real'] = df_metrics['label'].apply(lambda x: 'Humano' if x == 1 else 'IA')
df_metrics['Texto_ID'] = df_metrics.index
texts = df_metrics['text_cleaned']

# Diccionario para automatizar ejecución y registro de recursos
scoring_tasks = [
    ("Score_LLaDA_8B", batch_llada_scores, model_LLaDA_8B, tokenizer_LLaDA_8B, LLaDA_8B_DEVICE),
    ("Score_LLaDA_1_5B", batch_llada_scores, model_LLaDA_1_5B, tokenizer_LLaDA_1_5B, LLaDA_1_5B_DEVICE),
    ("Score_Dream_LLaDA_7B", batch_llada_scores, model_Dream_LLaDA_7B, tokenizer_Dream_LLaDA_7B, Dream_LLaDA_7B_DEVICE)
]

for col_name, func, model, tok, dev in scoring_tasks:
    print(f"\nCalculando {col_name}...")
    result, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    df_metrics[col_name] = result
    performance_data.append({
        "Modelo": col_name.split('_')[1],
        "Enfoque": "Sequence Score",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 1 completado ---")
df_metrics.head()


Calculando Score_LLaDA_8B...


LLaDA Scores: 100%|██████████| 250/250 [02:37<00:00,  1.58it/s]



Calculando Score_LLaDA_1_5B...


LLaDA Scores: 100%|██████████| 250/250 [02:38<00:00,  1.58it/s]



Calculando Score_Dream_LLaDA_7B...


LLaDA Scores: 100%|██████████| 250/250 [02:09<00:00,  1.93it/s]


--- Enfoque 1 completado ---


,text_cleaned,label,Clase_Real,Texto_ID,Score_LLaDA_8B,Score_LLaDA_1_5B,Score_Dream_LLaDA_7B
0,Never again...never again!!' This place is ter...,0,IA,0,-8.237720,-5.176360,-4.673721
1,"put the carpet on the floor, they measure it, ...",1,Humano,1,-3.692867,-1.451307,-5.033270
2,[substeps] You may do this process before you ...,1,Humano,2,-7.790140,-4.867730,-4.649153
3,"I believe mandatory minimum laws are unjust, c...",1,Humano,3,-12.524784,-11.683010,-1.815197
4,Wales coach Warren Gatland has hailed Shane Wi...,0,IA,4,-12.779110,-11.593557,-3.773181


### PAWN 

In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm


def calculate_five_metrics_diffusion(logits, labels, mask_positions):
    """
    Calcula las 5 métricas PAWN solo en las posiciones ENMASCARADAS.
    Basado en la lógica de reconstrucción de Language Diffusion.
    """
    B, T, V = logits.shape
    
    # Trabajamos con log_softmax para estabilidad numérica
    log_probs = F.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)
    
    # 1. Log-probabilidad del token real (¿Qué tan bien reconstruye el modelo el texto original?)
    log_p = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    
    # 2. Entropía (Incertidumbre del modelo en las zonas enmascaradas)
    entropy = -(probs * log_probs).sum(dim=-1)
    
    # 3. Max log-prob (Confianza máxima en la predicción)
    max_log_p, _ = log_probs.max(dim=-1)
    
    # ------------------------------------------------------------------
    # 🔥 CAMBIO CLAVE:
    # Rank y Top-p se calculan SOLO en tokens ENMASCARADOS
    # Evita crear tensores gigantes [B, T, V]
    # ------------------------------------------------------------------

    mask_float = mask_positions.float()
    num_masked = mask_float.sum(dim=1).clamp(min=1)

    # Extraemos únicamente las posiciones enmascaradas
    masked_log_probs = log_probs[mask_positions]   # [N_masked, V]
    masked_labels = labels[mask_positions]         # [N_masked]

    # 4. Rank normalizado (Posición ordinal del token real)
    token_log_p = masked_log_probs.gather(
        1, masked_labels.unsqueeze(1)
    )
    rank = (masked_log_probs > token_log_p).sum(dim=1).float() + 1.0
    rank_norm = rank / V

    # 5. Top-p (Masa de probabilidad acumulada necesaria para llegar al token real)
    token_prob = torch.exp(token_log_p)
    top_p = (torch.exp(masked_log_probs) >= token_prob).sum(dim=1).float() / V

    # Reconstruimos tensores [B, T] solo en posiciones enmascaradas
    rank_full = torch.zeros((B, T), device=logits.device)
    top_p_full = torch.zeros((B, T), device=logits.device)

    rank_full[mask_positions] = rank_norm
    top_p_full[mask_positions] = top_p

    results = []
    # Iteramos sobre las 5 métricas calculadas
    for M in [log_p, entropy, max_log_p, rank_full, top_p_full]:
        # Filtramos para obtener el promedio SOLO de los tokens que fueron enmascarados
        # Esto es lo que mide la capacidad de "denoising" o reconstrucción.
        avg_val = (M * mask_float).sum(dim=1) / num_masked
        results.append(avg_val.cpu().tolist())
        
    return results

def batch_diffusion_metrics(texts, model, tokenizer, batch_size, device, mask_ratio=0.35, num_samples=10):
    """
    Implementa el proceso de scoring por difusión para LLaDA.
    Aumentamos el mask_ratio a 0.20 para forzar al modelo a usar más contexto.
    Reducimos num_samples a 3 para balancear velocidad y estabilidad.
    """
    all_metrics = [[] for _ in range(5)]
    
    # Identificar token de máscara correcto
    if hasattr(tokenizer, 'mask_token_id') and tokenizer.mask_token_id is not None:
        mask_id = tokenizer.mask_token_id
    else:
        # Fallback para modelos que no tienen [MASK] definido explícitamente
        mask_id = tokenizer.vocab_size - 1 

    model.eval()
    
    # Determinamos el tipo de dato para autocast (bfloat16 para GPUs modernas como RTX 4500 Ada)
    dtype = torch.bfloat16 if device == 'cuda' else torch.float32

    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Diffusion Scoring"):
        batch_texts = texts[i:i+batch_size].tolist()
        # Acumulador para las muestras estocásticas de cada batch
        batch_accum = [[] for _ in range(5)]
        
        # Realizamos varias pasadas con diferentes máscaras para obtener un promedio robusto (Monte Carlo)
        for _ in range(num_samples):
            inputs = tokenizer(
                batch_texts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True, 
                max_length=512
            ).to(device)
            
            input_ids = inputs["input_ids"]
            att_mask = inputs["attention_mask"]
            B, T = input_ids.shape
            
            # Generamos máscara aleatoria excluyendo el padding
            mask_probs = torch.full((B, T), mask_ratio, device=device) * att_mask.float()
            
            # Opcional: Evitar enmascarar tokens especiales (si el tokenizer los tiene definidos)
            if hasattr(tokenizer, 'all_special_ids'):
                for special_id in tokenizer.all_special_ids:
                    mask_probs[input_ids == special_id] = 0.0

            mask_pos = torch.bernoulli(mask_probs).bool()
            
            # Garantizar que al menos un token esté enmascarado por secuencia
            for j in range(B):
                if not mask_pos[j].any():
                    valid_indices = att_mask[j].nonzero(as_tuple=True)[0]
                    if len(valid_indices) > 0:
                        random_idx = valid_indices[torch.randint(0, len(valid_indices), (1,))]
                        mask_pos[j, random_idx] = True

            # Crear la versión "corrupta" del texto
            corrupted_ids = input_ids.clone()
            corrupted_ids[mask_pos] = mask_id

            is_dream = model.__class__.__name__.lower().startswith("dream")

            with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
                if is_dream:
                    # 🔥 Dream NO acepta attention_mask 2D
                    outputs = model(input_ids=corrupted_ids)
                else:
                    outputs = model(
                        input_ids=corrupted_ids,
                        attention_mask=att_mask.to(torch.bool)
                    )

                
                # Extraer métricas solo de las posiciones que el modelo tuvo que reconstruir
                sample_m = calculate_five_metrics_diffusion(outputs.logits, input_ids, mask_pos)
                
                for m_idx in range(5):
                    batch_accum[m_idx].append(sample_m[m_idx])
        
        # Promediar las muestras para reducir el ruido de la selección aleatoria de máscaras
        for m_idx in range(5):
            avg_res = np.array(batch_accum[m_idx]).mean(axis=0)
            all_metrics[m_idx].extend(avg_res.tolist())
            
    return all_metrics

In [9]:
metrics_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

# Tareas de extracción PAWN
pawn_tasks = [
    ("LLaDA_8B", batch_diffusion_metrics, model_LLaDA_8B, tokenizer_LLaDA_8B, LLaDA_8B_DEVICE), # Usa tu función diffusion
    ("LLaDA_1_5B", batch_diffusion_metrics, model_LLaDA_1_5B, tokenizer_LLaDA_1_5B, LLaDA_1_5B_DEVICE), 
    ("Dream_LLaDA_7B", batch_diffusion_metrics, model_Dream_LLaDA_7B, tokenizer_Dream_LLaDA_7B, Dream_LLaDA_7B_DEVICE)
]

for mod_name, func, model, tok, dev in pawn_tasks:
    print(f"\nCalculando métricas PAWN para {mod_name}...")
    # Registramos recursos
    pawn_results, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    
    # Guardamos las 5 métricas en el DataFrame
    for idx, m_name in enumerate(metrics_names):
        df_metrics[f'{m_name}_{mod_name}'] = pawn_results[idx]
        
    # Añadimos a la tabla de rendimiento
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "PAWN Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 2: PAWN Metrics completado ---")
df_metrics.filter(like='Mlog_prob').head()


Calculando métricas PAWN para LLaDA_8B...


LLaDA Diffusion Scoring: 100%|██████████| 250/250 [42:43<00:00, 10.25s/it]



Calculando métricas PAWN para LLaDA_1_5B...


LLaDA Diffusion Scoring: 100%|██████████| 250/250 [42:42<00:00, 10.25s/it]



Calculando métricas PAWN para Dream_LLaDA_7B...


LLaDA Diffusion Scoring: 100%|██████████| 250/250 [38:17<00:00,  9.19s/it]


--- Enfoque 2: PAWN Metrics completado ---


,Mlog_prob_LLaDA_8B,Mlog_prob_LLaDA_1_5B,Mlog_prob_Dream_LLaDA_7B
0,-8.585288,-8.503447,-6.082106
1,-7.676292,-7.689149,-6.701238
2,-7.920280,-7.788554,-7.043788
3,-7.523477,-6.474917,-6.594908
4,-7.424961,-5.984822,-7.541761


### EMBEDINGS

In [10]:
def extract_cls_embeddings_diffusion(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de LLaDA (modelo de difusión).
    LLaDA procesa bidireccionalmente, similar a BERT, por lo que usamos
    el promedio de todos los tokens como representación global.
    
    Alternativa: pooling del primer token o mean pooling de toda la secuencia.
    """
    all_embeddings = []

    is_dream = model.__class__.__name__.lower().startswith("dream")
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings difusión"):
        batch = texts[i:i+batch_size].tolist()

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            if is_dream:
                # 🔥 Dream NO acepta attention_mask 2D
                outputs = model(
                    input_ids=inputs["input_ids"],
                    output_hidden_states=True
                )
            else:
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"].to(torch.bool),
                    output_hidden_states=True
                )

            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Mean pooling sobre tokens válidos (excluyendo padding)
            attention_mask = inputs["attention_mask"].unsqueeze(-1)
            masked_hidden = hidden_states * attention_mask
            sum_hidden = masked_hidden.sum(dim=1)
            count = attention_mask.sum(dim=1).clamp(min=1)

            mean_embedding = (sum_hidden / count).float().cpu().numpy()
            all_embeddings.extend(mean_embedding)
    
    return np.array(all_embeddings)


In [11]:
embedding_tasks = [
    ("LLaDA_8B", extract_cls_embeddings_diffusion, model_LLaDA_8B, tokenizer_LLaDA_8B, LLaDA_8B_DEVICE),
    ("LLaDA_1_5B", extract_cls_embeddings_diffusion, model_LLaDA_1_5B, tokenizer_LLaDA_1_5B, LLaDA_1_5B_DEVICE),
    ("Dream_LLaDA_7B", extract_cls_embeddings_diffusion, model_Dream_LLaDA_7B, tokenizer_Dream_LLaDA_7B, Dream_LLaDA_7B_DEVICE)
]

for mod_name, func, model, tok, dev in embedding_tasks:
    print(f"\n[INFO] Extrayendo embeddings para {mod_name}...")
    
    # 1. Ejecutamos la extracción y medimos recursos (Tiempo, VRAM, RAM)
    # Usamos BATCH_SIZE=4 como en tus otros scripts para evitar errores de memoria
    embs_matrix, t, v, ram = resource_wrapper(func, texts, model, tok, dev, BATCH_SIZE)
    
    # 2. Guardamos los embeddings en el DataFrame
    # Convertimos la matriz numpy a una lista para que cada celda sea un vector individual
    df_metrics[f'Embedding_{mod_name}'] = list(embs_matrix)
    
    # 3. Registramos el rendimiento en la tabla global de recursos
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "CLS Embedding Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

# Limpieza de caché de GPU
torch.cuda.empty_cache()

print("\n" + "="*50)
print("ESTADO DEL DATAFRAME INTEGRADO")
print("="*50)
# Mostramos las nuevas columnas creadas junto a las anteriores
cols_interes = ['Texto_ID'] + [c for c in df_metrics.columns if 'Embedding' in c or 'Score' in c or 'Mlog_prob' in c]
print(df_metrics[cols_interes].head())

# Guardado intermedio del progreso
df_metrics.to_csv('df_all_features_combined_LLADAs.csv', index=False)
print("\n✓ Todas las métricas (Scores, PAWN y Embeddings) guardadas en 'df_all_features_combined_LLADAs.csv'")


[INFO] Extrayendo embeddings para LLaDA_8B...


Extrayendo embeddings difusión: 100%|██████████| 250/250 [02:37<00:00,  1.59it/s]



[INFO] Extrayendo embeddings para LLaDA_1_5B...


Extrayendo embeddings difusión: 100%|██████████| 250/250 [02:37<00:00,  1.59it/s]



[INFO] Extrayendo embeddings para Dream_LLaDA_7B...


Extrayendo embeddings difusión: 100%|██████████| 250/250 [02:08<00:00,  1.95it/s]



ESTADO DEL DATAFRAME INTEGRADO
   Texto_ID  Score_LLaDA_8B  Score_LLaDA_1_5B  Score_Dream_LLaDA_7B  \
0         0       -8.237720         -5.176360             -4.673721   
1         1       -3.692867         -1.451307             -5.033270   
2         2       -7.790140         -4.867730             -4.649153   
3         3      -12.524784        -11.683010             -1.815197   
4         4      -12.779110        -11.593557             -3.773181   

   Mlog_prob_LLaDA_8B  Mlog_prob_LLaDA_1_5B  Mlog_prob_Dream_LLaDA_7B  \
0           -8.585288             -8.503447                 -6.082106   
1           -7.676292             -7.689149                 -6.701238   
2           -7.920280             -7.788554                 -7.043788   
3           -7.523477             -6.474917                 -6.594908   
4           -7.424961             -5.984822                 -7.541761   

                                  Embedding_LLaDA_8B  \
0  [-0.40302193, -0.44798902, -0.5466218, 0.33

In [12]:
df = pd.DataFrame(performance_data)

df.to_csv('df_performance_models_proachs_LLADAs.csv', index=False)
df.head()

,Modelo,Enfoque,Tiempo (s),VRAM Pico (GB),RAM usada (GB)
0,LLaDA,Sequence Score,157.957349,19.056250,0.102764
1,LLaDA,Sequence Score,158.279016,19.061132,0.003628
2,Dream,Sequence Score,129.575767,19.548054,0.007347
3,LLaDA_8B,PAWN Extraction,2563.025076,20.670338,-0.008617
4,LLaDA_1_5B,PAWN Extraction,2562.963023,20.668807,-0.001286


### CLASIFICACIÓN

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# =============================================================================
# 0. FUNCIÓN DE EVALUACIÓN (idéntica a Embedings.ipynb)
# =============================================================================
def evaluate_classifier(y_true, y_pred_probs):
    y_pred = (y_pred_probs >= 0.5).astype(int)
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics['ROC-AUC'] = roc_auc_score(y_true, y_pred_probs)
    except:
        metrics['ROC-AUC'] = np.nan
    return metrics

import ast

def parse_embedding_safe(x):
    """
    Convierte un string tipo '[0.1 0.2 ...]' o '[0.1, 0.2, ...]'
    en np.ndarray float32 de forma segura.
    """
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, str):
        # Normalizamos espacios -> comas
        x = x.replace('\n', ' ').replace('  ', ' ')
        if ',' not in x:
            x = x.replace(' ', ', ')
        return np.array(ast.literal_eval(x), dtype=np.float32)
    raise ValueError(f"Tipo inesperado en embedding: {type(x)}")

# =============================================================================
# 1. ARQUITECTURA Y MOTOR MLP (PAWN)
# =============================================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class DeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 2)
        )
    def forward(self, x): 
        return self.net(x)

def train_eval_mlp_full(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = DeepMLP(X.shape[1]).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()),
        batch_size=32, shuffle=True
    )

    for _ in range(100):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).float().to(DEVICE))
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

    return evaluate_classifier(y_test, probs)

# =============================================================================
# 2. CLASIFICACIÓN MACRO (SCORE / PAWN / EMBEDDINGS)
# =============================================================================
df = pd.read_csv("df_all_features_combined_LLADAs.csv")
y_all = df["label"].values
results_master = []

models_list = ['LLaDA_8B', 'LLaDA_1_5B', 'Dream_LLaDA_7B']
metrics_pawn_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

for m in models_list:
    print(f"\nEvaluando: {m}")

    # -------------------------------------------------------------------------
    # A) SCORE
    # -------------------------------------------------------------------------
    s_col = f"Score_{m}"
    if s_col in df.columns:
        X_s = df[[s_col]].values
        clfs = {
            'LogReg': LogisticRegression(max_iter=2000),
            'RandomForest': RandomForestClassifier(n_estimators=300, random_state=SEED),
            'XGBoost': xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, eval_metric="logloss", random_state=SEED),
            'XGB_Calib': CalibratedClassifierCV(
                xgb.XGBClassifier(eval_metric="logloss"), method="isotonic", cv=3
            )
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        for name, clf in clfs.items():
            cv_res = cross_validate(
                clf, X_s, y_all, cv=cv,
                scoring=['roc_auc', 'accuracy', 'f1', 'recall', 'precision']
            )
            results_master.append({
                'Modelo': m, 'Aprox': 'Score', 'Clf': name,
                'ROC-AUC': np.mean(cv_res['test_roc_auc']),
                'Accuracy': np.mean(cv_res['test_accuracy']),
                'F1': np.mean(cv_res['test_f1']),
                'Recall': np.mean(cv_res['test_recall']),
                'Precision': np.mean(cv_res['test_precision'])
            })

    # -------------------------------------------------------------------------
    # B) PAWN
    # -------------------------------------------------------------------------
    p_cols = [f"{met}_{m}" for met in metrics_pawn_names]
    if all(c in df.columns for c in p_cols):
        X_p = df[p_cols].values

        # PAWN + LogReg
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_p, y_all, test_size=0.2, stratify=y_all, random_state=SEED
        )
        sc = StandardScaler()
        clf_lr = LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=SEED
        ).fit(sc.fit_transform(X_tr), y_tr)
        p_lr = clf_lr.predict_proba(sc.transform(X_te))[:, 1]

        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'LogReg',
            **evaluate_classifier(y_te, p_lr)
        })

        # PAWN + DeepMLP
        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'DeepMLP',
            **train_eval_mlp_full(X_p, y_all)
        })

# -------------------------------------------------------------------------
# C) EMBEDDINGS (extracción + clasificación DIRECTA, sin guardar)
# -------------------------------------------------------------------------
print("\n" + "="*80)
print("EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)")
print("="*80)

# Diccionario que mapea modelo → función de extracción + objetos
embedding_configs = {
    'LLaDA_8B': {
        'extractor': lambda: extract_cls_embeddings_diffusion(
            texts, model_LLaDA_8B, tokenizer_LLaDA_8B, LLaDA_8B_DEVICE
        )
    },
        'LLaDA_1_5B': {
        'extractor': lambda: extract_cls_embeddings_diffusion(
            texts, model_LLaDA_1_5B, tokenizer_LLaDA_1_5B, LLaDA_1_5B_DEVICE
        )
    },    'Dream_LLaDA_7B': {
        'extractor': lambda: extract_cls_embeddings_diffusion(
            texts, model_Dream_LLaDA_7B, tokenizer_Dream_LLaDA_7B, Dream_LLaDA_7B_DEVICE
        )
    }
}

for m, cfg in embedding_configs.items():
    print(f"\n[EMB] {m} - Extrayendo embeddings...")
    
    # 1. Extracción REAL del embedding (np.ndarray)
    X_e = cfg['extractor']()
    print(f"   Shape: {X_e.shape}")

    # 2. Split (idéntico a Embedings.ipynb)
    X_train, X_test, y_train, y_test = train_test_split(
        X_e, y_all, test_size=0.2, random_state=SEED, stratify=y_all
    )

    # 3. Escalado
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Clasificador lineal
    clf = LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        class_weight='balanced'
    )
    clf.fit(X_train_scaled, y_train)

    # 5. Predicción
    y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

    # 6. Métricas
    metrics = evaluate_classifier(y_test, y_pred_probs)

    print(f"   Resultados: {metrics}")

    # 7. Inserción DIRECTA en la macro-tabla
    results_master.append({
        'Modelo': m,
        'Aprox': 'Embedding',
        'Clf': 'LogReg',
        'ROC-AUC': metrics['ROC-AUC'],
        'Accuracy': metrics['Accuracy'],
        'F1': metrics['F1'],
        'Recall': metrics['Recall'],
        'Precision': metrics['Precision']
    })

# =============================================================================
# 3. PRESENTACIÓN DE RESULTADOS
# =============================================================================
df_final = pd.DataFrame(results_master)
print("\n" + "="*120)
print("MACRO TABLA DE RESULTADOS (COMPARATIVA FINAL)")
print("="*120)
display(df_final.set_index(['Modelo', 'Aprox', 'Clf']).round(4))



Evaluando: LLaDA_8B

Evaluando: LLaDA_1_5B

Evaluando: Dream_LLaDA_7B

EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)

[EMB] LLaDA_8B - Extrayendo embeddings...


Extrayendo embeddings difusión: 100%|██████████| 125/125 [02:39<00:00,  1.27s/it]


   Shape: (1000, 4096)
   Resultados: {'Accuracy': 0.82, 'Precision': 0.8369565217391305, 'Recall': 0.7857142857142857, 'F1': 0.8105263157894738, 'ROC-AUC': 0.9101640656262505}

[EMB] LLaDA_1_5B - Extrayendo embeddings...


Extrayendo embeddings difusión: 100%|██████████| 125/125 [02:39<00:00,  1.28s/it]


   Shape: (1000, 4096)
   Resultados: {'Accuracy': 0.76, 'Precision': 0.7551020408163265, 'Recall': 0.7551020408163265, 'F1': 0.7551020408163265, 'ROC-AUC': 0.8485394157663065}

[EMB] Dream_LLaDA_7B - Extrayendo embeddings...


Extrayendo embeddings difusión: 100%|██████████| 125/125 [02:05<00:00,  1.00s/it]


   Shape: (1000, 3584)
   Resultados: {'Accuracy': 0.82, 'Precision': 0.7818181818181819, 'Recall': 0.8775510204081632, 'F1': 0.8269230769230769, 'ROC-AUC': 0.908563425370148}

MACRO TABLA DE RESULTADOS (COMPARATIVA FINAL)


ROC-AUC  Accuracy      F1  Recall  \
Modelo         Aprox     Clf                                               
LLaDA_8B       Score     LogReg         0.5392     0.528  0.4697  0.4295   
                         RandomForest   0.5646     0.544  0.5382  0.5459   
                         XGBoost        0.5547     0.530  0.5296  0.5461   
                         XGB_Calib      0.5543     0.537  0.5840  0.6975   
               PAWN      LogReg         0.5298     0.495  0.4925  0.5000   
                         DeepMLP        0.6481     0.615  0.6316  0.6735   
LLaDA_1_5B     Score     LogReg         0.5710     0.548  0.4867  0.4417   
                         RandomForest   0.5145     0.487  0.4704  0.4662   
                         XGBoost        0.5348     0.506  0.4985  0.5032   
                         XGB_Calib      0.5323     0.512  0.4906  0.5095   
               PAWN      LogReg         0.6037     0.535  0.5181  0.5102   
                         DeepMLP        0.7080     0.650  0.6903  0.7959   
Dream_LLaDA_7B Score     LogReg         0.5360     0.508  0.4561  0.4255   
                         RandomForest   0.5528     0.518  0.5024  0.4989   
                         XGBoost        0.5691     0.527  0.5057  0.4970   
                         XGB_Calib      0.5767     0.567  0.5745  0.6345   
               PAWN      LogReg         0.5726     0.550  0.5408  0.5408   
                         DeepMLP        0.6758     0.615  0.6207  0.6429   
LLaDA_8B       Embedding LogReg         0.9102     0.820  0.8105  0.7857   
LLaDA_1_5B     Embedding LogReg         0.8485     0.760  0.7551  0.7551   
Dream_LLaDA_7B Embedding LogReg         0.9086     0.820  0.8269  0.8776   

                                       Precision  
Modelo         Aprox     Clf                      
LLaDA_8B       Score     LogReg           0.5228  
                         RandomForest     0.5322  
                         XGBoost          0.5167  
                         XGB_Calib        0.5193  
               PAWN      LogReg           0.4851  
                         DeepMLP          0.5946  
LLaDA_1_5B     Score     LogReg           0.5465  
                         RandomForest     0.4756  
                         XGBoost          0.4954  
                         XGB_Calib        0.5161  
               PAWN      LogReg           0.5263  
                         DeepMLP          0.6094  
Dream_LLaDA_7B Score     LogReg           0.4940  
                         RandomForest     0.5063  
                         XGBoost          0.5166  
                         XGB_Calib        0.5514  
               PAWN      LogReg           0.5408  
                         DeepMLP          0.6000  
LLaDA_8B       Embedding LogReg           0.8370  
LLaDA_1_5B     Embedding LogReg           0.7551  
Dream_LLaDA_7B Embedding LogReg           0.7818

In [14]:
df_final.to_csv('df_results_models_proachs_LLADAs.csv', index=False)